# Ekstraksi PDF Putusan MK — Section-Based
**Tujuan:** Mengekstrak teks putusan Mahkamah Konstitusi secara otomatis berdasarkan section (kepala, pihak, duduk perkara, pertimbangan, amar, penutup) untuk keperluan anotasi NER di Doccano.

**Output:** File `.jsonl` siap upload ke Doccano.

---
### Urutan Menjalankan Cell:
1. **Cell 1** — Install library
2. **Cell 2** — Import & konfigurasi section
3. **Cell 3** — Fungsi utilitas
4. **Cell 4** — Fungsi utama ekstraksi
5. **Cell 5** — Fungsi output
6. **Cell 6** — Upload PDF & jalankan ekstraksi
7. **Cell 7** — Simpan hasil & download
8. **Cell 8** *(opsional)* — Cek isi section tertentu
9. **Cell 9** *(opsional)* — Sesuaikan pola regex
9. **Cell 9** *(opsional)* — Sesuaikan pola regex
10. **Cell 10** *(opsional)* — Debug section pihak tidak terdeteksi


In [ ]:
# ================================================================
# CELL 1 — INSTALASI LIBRARY
# ================================================================
# Jalankan cell ini pertama kali, lalu restart runtime jika diminta

# !pip install pdfplumber pypdf -q

In [ ]:
import os
import re
import json
import logging
from pathlib import Path
from typing import Optional
from google.colab import files

import pdfplumber
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

# ================================================================
# KONFIGURASI SECTION — disesuaikan dengan format putusan MK
#
# Temuan dari analisis PDF 27/PUU-XVII/2019:
#
#   KEPALA  : dimulai dari 'PUTUSAN\nNomor ...'
#   PIHAK   : TIDAK ada header 'PEMOHON' — dimulai dari
#             frasa 'diajukan oleh:' lalu langsung '1. Nama :'
#             Kuasa hukum: 'memberi kuasa kepada'
#             Penutup pihak: 'Selanjutnya Pemohon ... disebut'
#   DUDUK   : dimulai dari '2. DUDUK PERKARA' atau '[2.1]'
#   PERTIMB : dimulai dari '3. PERTIMBANGAN HUKUM' atau '[3.1]'
#   AMAR    : dimulai dari '5. AMAR PUTUSAN' atau 'Mengadili:'
#   PENUTUP : dimulai dari 'Demikian diputus dalam Rapat'
# ================================================================

SECTION_CONFIG = {

    # ── KEPALA ─────────────────────────────────────────────────
    # Hanya sampai sebelum 'diajukan oleh:' (pihak mulai di sana)
    "kepala": {
        "patterns": [
            r"PUTUSAN\nNomor\s+[\d]+/PUU",
            r"PUTUSAN\s+Nomor\s+[\d]+/PUU",
        ],
        "max_chars": 600,   # singkat — hanya sampai sebelum 'diajukan oleh:'
        "deskripsi": "Kepala putusan (nomor, MK, konteks UU)",
        "entitas": ["CASE_NUMBER", "ORG", "LAW_NAME"],
    },

    # ── PIHAK ──────────────────────────────────────────────────
    # Pola utama: 'diajukan oleh:' → langsung '1. Nama :'
    # Mencakup: pemohon (PARTY), kuasa hukum (LAWYER), termohon (ORG)
    "pihak": {
        "patterns": [
            r"diajukan oleh:\n1\. Nama",          # ← pola utama dari PDF ini
            r"diajukan oleh:\s*\n\s*\d+\.\s*Nama", # variasi spasi
            r"diajukan oleh:",                     # fallback jika format sedikit beda
            r"yang diajukan oleh:",
            r"[Dd]iajukan\s+oleh\s*:",
        ],
        "max_chars": 3000,
        "deskripsi": "Para pihak: pemohon, kuasa hukum, termohon",
        "entitas": ["PARTY", "LAWYER", "ORG", "DATE"],
    },

    # ── DUDUK PERKARA ──────────────────────────────────────────
    # Format: '2. DUDUK PERKARA' atau '[2.1] Menimbang ...'
    "duduk_perkara": {
        "patterns": [
            r"\d+\.\s*DUDUK\s+PERKARA",
            r"DUDUK\s+PERKARA",
            r"\[2\.1\]\s*Menimbang",
        ],
        "max_chars": 1500,
        "deskripsi": "Duduk perkara — paragraf awal (peristiwa, UU, tanggal)",
        "entitas": ["EVENT", "LAW_NAME", "LEGAL_REF", "DATE", "CASE_NUMBER"],
    },

    # ── PERTIMBANGAN HUKUM ─────────────────────────────────────
    # Format: '3. PERTIMBANGAN HUKUM' atau '[3.1] Menimbang ...'
    "pertimbangan": {
        "patterns": [
            r"\d+\.\s*PERTIMBANGAN\s+HUKUM",
            r"PERTIMBANGAN\s+HUKUM",
            r"\[3\.1\]\s*Menimbang",
        ],
        "max_chars": 1200,
        "deskripsi": "Pertimbangan hukum — paragraf pembuka",
        "entitas": ["LAW_NAME", "LEGAL_REF", "ORG", "CASE_NUMBER"],
    },

    # ── AMAR PUTUSAN ───────────────────────────────────────────
    # Format: '5. AMAR PUTUSAN' → 'Mengadili:' → kalimat dictum
    "amar": {
        "patterns": [
            r"\d+\.\s*AMAR\s+PUTUSAN",
            r"AMAR\s+PUTUSAN",
            r"Mengadili\s*:",
        ],
        "max_chars": 2000,
        "deskripsi": "Amar putusan (diktum resmi MK)",
        "entitas": ["DICTUM", "LAW_NAME", "LEGAL_REF"],
    },

    # ── PENUTUP ────────────────────────────────────────────────
    # Format: 'Demikian diputus dalam Rapat Permusyawaratan Hakim'
    # berisi nama hakim, tanggal, tempat sidang, panitera
    "penutup": {
        "patterns": [
            r"Demikian diputus dalam Rapat Permusyawaratan Hakim",
            r"[Dd]emikian diputus(?:kan)?",
            r"[Dd]iucapkan dalam [Ss]idang [Pp]leno",
        ],
        "max_chars": 2500,
        "deskripsi": "Penutup & tanda tangan hakim",
        "entitas": ["JUDGE", "DATE", "EVENT", "ORG"],
    },
}

SECTION_ORDER = ["kepala", "pihak", "duduk_perkara", "pertimbangan", "amar", "penutup"]

print('✅ Konfigurasi berhasil dimuat.')
print(f'   Total section: {len(SECTION_CONFIG)}')
for k, v in SECTION_CONFIG.items():
    print(f'   • {k:20s} → {v["deskripsi"]}')


In [ ]:
# ================================================================
# CELL 3 — FUNGSI UTILITAS
# ================================================================

def clean_text(text: str) -> str:
    """Normalkan teks hasil ekstraksi PDF (tahap awal)."""
    if not text:
        return ""
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_for_doccano(text: str) -> str:
    """
    Bersihkan teks untuk tampilan optimal di Doccano:

    1. Hapus garis putus (--- dst) yang muncul sebagai
       pemisah visual di putusan MK
       Contoh: 'Sebagai ------- Pemohon I' → 'Sebagai Pemohon I'

    2. Perbaiki soft-hyphen: kata terpotong di akhir baris PDF
       Contoh: 'Undang-\nUndang' → 'Undang-Undang'
       Contoh: 'menghalang-\nhalangi' → 'menghalang-halangi'

    3. Gabungkan baris yang terpotong di tengah kalimat
       menjadi satu baris panjang, KECUALI:
       - Baris berikutnya = header KAPITAL (nama section)
       - Baris berikutnya = penomoran (1. 2. A. B.)
       - Baris berikutnya = penanda paragraf ([1.1], [2.1])
       - Baris sebelumnya diakhiri tanda baca kuat (. ; :)

    4. Tanda hubung kata (Undang-Undang, sendiri-sendiri,
       perundang-undangan) TIDAK terganggu.
    """

    # STEP 1: Hapus garis putus (3 atau lebih tanda hubung berurutan)
    text = re.sub(r"\s*-{3,}\s*", " ", text)

    # STEP 2: Perbaiki soft-hyphen (kata terpotong di akhir baris)
    # Pola: huruf + tanda hubung + newline + huruf
    text = re.sub(r"(\w)-\n(\w)", r"\1-\2", text)

    # STEP 3: Gabungkan baris terpotong di tengah kalimat
    def _merge_line(m):
        before = m.group(1)
        after  = m.group(2)

        # Pertahankan \n sebelum header KAPITAL SEMUA (section title)
        if re.match(r"^[A-Z][A-Z\s]{5,}$", after.strip()):
            return before + "\n" + after

        # Pertahankan \n sebelum penomoran (1. 2. 3. atau A. B. C.)
        if re.match(r"^\d+\.\s|^[A-Z]\.\s", after.strip()):
            return before + "\n" + after

        # Pertahankan \n sebelum penanda paragraf [x.x]
        if re.match(r"^\[[\d\.]+\]", after.strip()):
            return before + "\n" + after

        # Pertahankan \n setelah tanda baca kuat (. ; :)
        if re.search(r"[\.;:]\s*$", before):
            return before + "\n" + after

        # Gabungkan dengan spasi
        return before + " " + after

    # Jalankan 2x untuk menangani baris sangat pendek berurutan
    text = re.sub(r"([^\n])\n([^\n])", _merge_line, text)
    text = re.sub(r"([^\n])\n([^\n])", _merge_line, text)

    # STEP 4: Bersihkan spasi berlebihan
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_full_text(pdf_path: str) -> tuple:
    """
    Ekstrak seluruh teks dari PDF menggunakan pdfplumber.
    Fallback ke pypdf jika pdfplumber gagal.
    Return: (full_text, page_count)
    """
    full_text = ""
    page_count = 0
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page_count = len(pdf.pages)
            parts = []
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    parts.append(t)
            full_text = "\n".join(parts)
        log.info(f"  pdfplumber OK — {page_count} halaman, {len(full_text)} karakter")
    except Exception as e:
        log.warning(f"  pdfplumber gagal ({e}), mencoba pypdf...")
        try:
            reader = PdfReader(pdf_path)
            page_count = len(reader.pages)
            parts = [p.extract_text() or "" for p in reader.pages]
            full_text = "\n".join(parts)
            log.info(f"  pypdf OK — {page_count} halaman, {len(full_text)} karakter")
        except Exception as e2:
            log.error(f"  Ekstraksi gagal: {e2}")
    return clean_text(full_text), page_count


def find_section(full_text: str, key: str) -> str | None:
    """Temukan section dan kembalikan teks yang sudah dibersihkan."""
    cfg      = SECTION_CONFIG[key]
    max_chars = cfg["max_chars"]
    for pattern in cfg["patterns"]:
        m = re.search(pattern, full_text, re.IGNORECASE | re.MULTILINE)
        if m:
            raw = full_text[m.start(): m.start() + max_chars]
            return clean_for_doccano(raw)   # ← teks langsung dibersihkan
    return None


def extract_metadata(text: str, filename: str = "") -> dict:
    """Ekstrak metadata dasar dari teks putusan."""
    meta = {"source_file": filename}
    m = re.search(
        r"(?:Nomor|No\.?)\s+([\d]+/(?:PUU|PHPU|PHP|PKE|SKLN|MK)-[IVXLCDM]+/\d{4})",
        text, re.IGNORECASE
    )
    if m:
        meta["case_number"] = m.group(1).strip()
    m = re.search(
        r"(?:diucapkan|ditetapkan|pada\s+(?:hari\s+\w+,?\s+)?tanggal)\s*"
        r"(\d{1,2}\s+\w+\s+\d{4})",
        text, re.IGNORECASE
    )
    if m:
        meta["date"] = m.group(1).strip()
    if re.search(r"Pengujian\s+Undang-Undang|/PUU-", text, re.IGNORECASE):
        meta["jenis_perkara"] = "PUU"
    elif re.search(r"PHPU|Perselisihan\s+Hasil\s+Pemilihan\s+Umum", text, re.IGNORECASE):
        meta["jenis_perkara"] = "PHPU"
    elif re.search(r"SKLN|Sengketa\s+Kewenangan\s+Lembaga\s+Negara", text, re.IGNORECASE):
        meta["jenis_perkara"] = "SKLN"
    else:
        meta["jenis_perkara"] = "lainnya"
    return meta


print("✅ Fungsi utilitas siap.")
print("   clean_text()        — normalisasi awal")
print("   clean_for_doccano() — hapus garis putus, gabung baris, jaga tanda hubung")
print("   extract_full_text() — ekstrak PDF")
print("   find_section()      — deteksi & bersihkan section")
print("   extract_metadata()  — nomor putusan, tanggal, jenis")


In [ ]:
def process_pdf(pdf_path: str, verbose: bool = True) -> dict:
    """
    Proses satu file PDF putusan MK dengan section-based extraction.

    Return dict:
      - filename, page_count
      - sections        : dict {key: teks_section}
      - sections_found  : list section yang berhasil ditemukan
      - sections_missing: list section yang tidak ditemukan
      - text            : teks gabungan siap anotasi Doccano
      - meta            : metadata putusan
    """
    path = Path(pdf_path)
    if verbose:
        print(f"\n{'='*60}")
        print(f"📄 File : {path.name}")
        print(f"{'='*60}")

    # ── Ekstrak teks penuh ─────────────────────────────────────
    full_text, page_count = extract_full_text(pdf_path)

    if not full_text.strip():
        log.error("Teks kosong — periksa file PDF.")
        return {}

    # ── Ekstrak tiap section ───────────────────────────────────
    sections = {}
    found = []
    missing = []

    for key in SECTION_ORDER:
        result = find_section(full_text, key)
        sections[key] = result
        if result:
            found.append(key)
        else:
            missing.append(key)

    # ── Laporan per file ───────────────────────────────────────
    if verbose:
        print(f"\n📊 Hasil Deteksi Section:")
        print(f"   {'Section':<20} {'Status':<10} {'Entitas Target'}")
        print(f"   {'-'*60}")
        for key in SECTION_ORDER:
            cfg   = SECTION_CONFIG[key]
            ok    = "✅ Ditemukan" if sections[key] else "❌ Tidak ada"
            chars = f"({len(sections[key])} kar)" if sections[key] else ""
            ents  = ", ".join(cfg["entitas"])
            print(f"   {key:<20} {ok:<15} {chars:<12} {ents}")

        pct = len(found) / len(SECTION_ORDER) * 100
        print(f"\n   Total: {len(found)}/{len(SECTION_ORDER)} section ({pct:.0f}%)")

        if missing:
            print(f"\n⚠️  Section tidak ditemukan: {', '.join(missing)}")
            print(f"   → Cek pola regex di SECTION_CONFIG atau format putusan berbeda.")

    # ── Gabungkan section untuk Doccano ───────────────────────
    parts = []
    for key in SECTION_ORDER:
        if sections[key]:
            header = f"=== {key.upper()} ===\n"
            parts.append(header + sections[key])

    combined_text = "\n\n".join(parts)

    # ── Metadata ───────────────────────────────────────────────
    meta = extract_metadata(full_text, path.name)

    if verbose:
        print(f"\n📋 Metadata:")
        for k, v in meta.items():
            print(f"   {k:<20}: {v}")
        print(f"\n📝 Total teks untuk anotasi: {len(combined_text)} karakter")

    return {
        "filename":         path.name,
        "page_count":       page_count,
        "sections":         sections,
        "sections_found":   found,
        "sections_missing": missing,
        "text":             combined_text,
        "meta":             meta,
    }


def process_batch(pdf_paths: list, verbose: bool = True) -> list:
    """Proses beberapa file PDF sekaligus."""
    all_records = []
    print(f"\n🗂️  Memproses {len(pdf_paths)} file PDF...\n")

    for i, path in enumerate(pdf_paths, 1):
        print(f"[{i}/{len(pdf_paths)}]", end=" ")
        try:
            rec = process_pdf(path, verbose=verbose)
            if rec and rec.get("text"):
                all_records.append(rec)
            else:
                print(f"⚠️  Dilewati (teks kosong): {Path(path).name}")
        except Exception as e:
            print(f"❌ Error: {Path(path).name} — {e}")

    print(f"\n✅ Berhasil: {len(all_records)}/{len(pdf_paths)} file")
    return all_records


print("✅ Fungsi utama ekstraksi siap.")

In [ ]:
def save_jsonl(records: list, output_path: str) -> None:
    """
    Simpan hasil ekstraksi ke format JSONL siap upload Doccano.
    Setiap baris = satu dokumen putusan.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for rec in records:
            row = {
                "text": rec["text"],
                "meta": rec["meta"],
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"💾 JSONL disimpan: {output_path} ({len(records)} dokumen)")


def save_txt_preview(records: list, output_path: str) -> None:
    """
    Simpan preview teks per section ke file .txt
    untuk memudahkan pengecekan sebelum anotasi.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for i, rec in enumerate(records, 1):
            f.write(f"{'#'*70}\n")
            f.write(f"DOKUMEN {i}: {rec['filename']}\n")
            f.write(f"No. Putusan : {rec['meta'].get('case_number', '-')}\n")
            f.write(f"Tanggal     : {rec['meta'].get('date', '-')}\n")
            f.write(f"Jenis       : {rec['meta'].get('jenis_perkara', '-')}\n")
            f.write(f"{'#'*70}\n\n")
            f.write(rec["text"])
            f.write("\n\n")
    print(f"📄 Preview disimpan: {output_path}")


def save_report(records: list, output_path: str) -> None:
    """Simpan laporan diagnostik ringkas."""
    lines = ["=" * 70, "LAPORAN EKSTRAKSI PUTUSAN MK", "=" * 70, ""]
    ok_total = 0

    for i, rec in enumerate(records, 1):
        found   = rec.get("sections_found", [])
        missing = rec.get("sections_missing", [])
        pct     = len(found) / len(SECTION_ORDER) * 100 if SECTION_ORDER else 0
        ok_total += 1 if pct == 100 else 0

        lines.append(f"[{i}] {rec['filename']}")
        lines.append(f"    Halaman     : {rec.get('page_count', '-')}")
        lines.append(f"    No. Putusan : {rec['meta'].get('case_number', '-')}")
        lines.append(f"    Tanggal     : {rec['meta'].get('date', '-')}")
        lines.append(f"    Jenis       : {rec['meta'].get('jenis_perkara', '-')}")
        lines.append(f"    Section OK  : {len(found)}/{len(SECTION_ORDER)} ({pct:.0f}%)")
        lines.append(f"    Ditemukan   : {', '.join(found) or '-'}")
        lines.append(f"    Tidak ada   : {', '.join(missing) or '-'}")
        lines.append(f"    Karakter    : {len(rec.get('text', ''))}")
        lines.append("")

    lines += [
        "=" * 70,
        f"RINGKASAN: {len(records)} dokumen diproses",
        f"           {ok_total} dokumen semua section lengkap",
        f"           {len(records) - ok_total} dokumen ada section tidak ditemukan",
        "=" * 70,
    ]

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"📋 Laporan disimpan: {output_path}")


def download_all(paths: list) -> None:
    """Download semua file output dari Colab ke komputer lokal."""
    for p in paths:
        if os.path.exists(p):
            files.download(p)
            print(f"⬇️  Download: {p}")
        else:
            print(f"⚠️  File tidak ditemukan: {p}")


print("✅ Fungsi output siap.")

In [ ]:
#
# PILIHAN A: Upload file satu per satu dari komputer lokal
# --------------------------------------------------------
# uploaded = files.upload()
# pdf_paths = list(uploaded.keys())
#
#
# PILIHAN B: Upload ke Google Drive lalu mount
# --------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# folder = "/content/drive/MyDrive/putusan_mk/"
# pdf_paths = sorted(str(p) for p in Path(folder).glob("*.pdf"))
#
#
# PILIHAN C: Upload manual ke Colab Files panel (/content/)
# ----------------------------------------------------------
# pdf_paths = sorted(str(p) for p in Path("/content/").glob("*.pdf"))

# ── Jalankan ekstraksi ─────────────────────────────────────────
# Ganti dengan salah satu pilihan di atas sesuai kebutuhan Anda

# Contoh untuk satu file:
# records = [process_pdf("nama_file.pdf")]

# Contoh untuk banyak file:
# records = process_batch(pdf_paths)

In [ ]:
# Jalankan setelah Cell 6 selesai

# output_dir = "/content/output_mk"
# os.makedirs(output_dir, exist_ok=True)
#
# jsonl_path   = f"{output_dir}/doccano_dataset.jsonl"
# preview_path = f"{output_dir}/preview_teks.txt"
# report_path  = f"{output_dir}/laporan_ekstraksi.txt"
#
# save_jsonl(records, jsonl_path)
# save_txt_preview(records, preview_path)
# save_report(records, report_path)
#
# # Download semua file ke komputer lokal
# download_all([jsonl_path, preview_path, report_path])

In [ ]:
#
# Untuk mengecek isi section tertentu dari satu dokumen:
#
# rec = records[0]   # dokumen pertama
# key = "pihak"      # ganti dengan section yang ingin dilihat
#
# print(f"=== Section: {key.upper()} ===")
# print(rec["sections"].get(key, "— Section tidak ditemukan —"))

In [ ]:
#
# Jika ada section yang tidak terdeteksi, tambahkan pola baru
# di SECTION_CONFIG pada Cell 2, contoh:
#
# SECTION_CONFIG["pihak"]["patterns"].append(
#     r"Nama\s+Pemohon\s*:"       # pola tambahan
# )
#
# Untuk menguji pola baru pada teks tertentu:
#
# import re
# test_text = records[0]["sections"].get("kepala", "")
# pattern   = r"Nomor\s+[\d]+/PUU"
# match     = re.search(pattern, test_text, re.IGNORECASE)
# print("Match:", match.group() if match else "Tidak ditemukan")

In [ ]:
# ================================================================
# CELL 10 — DEBUG SECTION PIHAK
# Jalankan jika section 'pihak' masih tidak terdeteksi.
# Cell ini menampilkan teks mentah di sekitar frasa pemisah
# sehingga Anda bisa menyesuaikan pola regex secara tepat.
# ================================================================

def debug_section_pihak(pdf_path: str, window: int = 300) -> None:
    """
    Tampilkan konteks teks di sekitar kata kunci yang berkaitan
    dengan section pihak: 'mengajukan', 'Nama :', 'Pekerjaan :'.
    Gunakan output ini untuk menentukan pola regex yang tepat.
    """
    full_text, _ = extract_full_text(pdf_path)

    kata_kunci = [
        r"mengajukan\s+permohonan",
        r"selanjutnya\s+disebut",
        r"^\s*Nama\s*:",
        r"^\s*1\.\s*Nama",
        r"^\s*Pekerjaan\s*:",
        r"^\s*Alamat\s*:",
    ]

    print(f"{'='*65}")
    print(f"DEBUG SECTION PIHAK: {Path(pdf_path).name}")
    print(f"{'='*65}")

    ditemukan = False
    for kw in kata_kunci:
        m = re.search(kw, full_text, re.IGNORECASE | re.MULTILINE)
        if m:
            ditemukan = True
            start = max(0, m.start() - 80)
            end   = min(len(full_text), m.end() + window)
            print(f"\n✅ Kata kunci ditemukan: '{kw}'")
            print(f"   Posisi karakter: {m.start()}")
            print(f"\n--- Konteks teks ({window} karakter setelah match) ---")
            print(repr(full_text[start:end]))
            print(f"---")
        else:
            print(f"\n❌ Tidak ditemukan: '{kw}'")

    if not ditemukan:
        print("\n⚠️  Tidak ada kata kunci yang cocok.")
        print("    Tampilkan 500 karakter awal teks untuk inspeksi manual:")
        print(repr(full_text[:500]))

    print(f"\n{'='*65}")
    print("CARA PAKAI HASIL DI ATAS:")
    print("  1. Lihat teks di sekitar kata kunci yang ✅ ditemukan")
    print("  2. Identifikasi frasa unik TEPAT SEBELUM daftar Nama/Pekerjaan")
    print("  3. Tambahkan pola baru di Cell 2:")
    print("     SECTION_CONFIG['pihak']['patterns'].insert(0, r'pola_baru')")
    print(f"{'='*65}")


# Jalankan debug untuk satu file:
# debug_section_pihak("nama_file.pdf")
#
# Atau dari records yang sudah diproses:
# debug_section_pihak(records[0]['meta']['source_file'])
